# DataOps Pipeline — Validation

Validates the DataOps pipeline output by querying the Glue Catalog
table `campaign_results` via Athena and checking row counts,
schema, and sample data.

In [11]:
# Parameters
catalog_db = "stack_bank_mktg_dev"
table_name = "campaign_results"
region = "us-east-2"
# Leave empty to derive from the Glue table S3 location
athena_output = ""

In [12]:
import boto3
import time
import os

catalog_db = catalog_db or os.environ.get('ATHENA_DB_NAME', 'stack_bank_mktg_dev')
region = region or os.environ.get('AWS_DEFAULT_REGION', 'us-east-2')

athena = boto3.client('athena', region_name=region)
glue = boto3.client('glue', region_name=region)

print(f'Database: {catalog_db}')
print(f'Table:    {table_name}')
print(f'Region:   {region}')

Database: stack_bank_mktg_dev
Table:    campaign_results
Region:   us-east-2


## 1. Check Table Exists in Glue Catalog

In [13]:
try:
    resp = glue.get_table(DatabaseName=catalog_db, Name=table_name)
    table = resp['Table']
    table_location = table['StorageDescriptor']['Location'].rstrip('/')
    print(f'Table found: {catalog_db}.{table_name}')
    print(f'Location:    {table_location}')
    print(f'Format:      {table["StorageDescriptor"]["InputFormat"]}')
    print(f'Columns:     {len(table["StorageDescriptor"]["Columns"])}')
    print()
    for col in table['StorageDescriptor']['Columns']:
        print(f'  {col["Name"]:25s} {col["Type"]}')
except glue.exceptions.EntityNotFoundException:
    raise RuntimeError(f'Table {catalog_db}.{table_name} not found')

# Match data_pipeline.yaml: .../output/athena_results/
if not athena_output:
    output_root = table_location.rsplit('/', 1)[0]  # .../output
    athena_output = f'{output_root}/athena_results/'
print(f'Athena output: {athena_output}')

Table found: stack_bank_mktg_dev.campaign_results
Location:    s3://amazon-sagemaker-442460399726-us-east-2-a1501210c441/dzd-5e6vtm4oytol49/6d3pq5vm70uxm1/shared/stack/data_pipeline/output/transformed
Format:      org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat
Columns:     23

  age                       int
  job                       string
  marital                   string
  education                 string
  default                   string
  housing                   string
  loan                      string
  contact                   string
  month                     string
  day_of_week               string
  duration                  int
  campaign                  int
  pdays                     int
  previous                  int
  poutcome                  string
  emp_var_rate              double
  cons_price_idx            double
  cons_conf_idx             double
  euribor3m                 double
  nr_employed               double
  y                   

## 2. Row Count

In [14]:
def run_athena_query(query, database, output_location):
    """Run Athena query and return results."""
    resp = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location},
    )
    qid = resp['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    if state != 'SUCCEEDED':
        reason = status['QueryExecution']['Status'].get('StateChangeReason', '')
        raise RuntimeError(f'Query failed: {state} {reason}'.strip())
    results = athena.get_query_results(QueryExecutionId=qid)
    return results

results = run_athena_query(
    f'SELECT COUNT(*) as cnt FROM {table_name}', catalog_db, athena_output
)
row_count = int(results['ResultSet']['Rows'][1]['Data'][0]['VarCharValue'])
print(f'Row count: {row_count}')
assert row_count > 0, 'Table is empty'

Row count: 40846


## 3. Sample Data

In [15]:
results = run_athena_query(f'SELECT * FROM {table_name} LIMIT 5', catalog_db, athena_output)
headers = [col['VarCharValue'] for col in results['ResultSet']['Rows'][0]['Data']]
print(' | '.join(headers))
print('-' * 80)
for row in results['ResultSet']['Rows'][1:]:
    vals = [d.get('VarCharValue', '') for d in row['Data']]
    print(' | '.join(vals))

age | job | marital | education | default | housing | loan | contact | month | day_of_week | duration | campaign | pdays | previous | poutcome | emp_var_rate | cons_price_idx | cons_conf_idx | euribor3m | nr_employed | y | age_group | contacted_before
--------------------------------------------------------------------------------
57 | services | married | high.school | unknown | no | no | telephone | may | mon | 149 | 1 | 999 | 0 | nonexistent | 1.1 | 93.994 | -36.4 | 4.857 | 5191.0 | no | senior | 
40 | admin. | married | basic.6y | no | no | no | telephone | may | mon | 151 | 1 | 999 | 0 | nonexistent | 1.1 | 93.994 | -36.4 | 4.857 | 5191.0 | no | middle | 
29 | blue-collar | single | high.school | no | no | yes | telephone | may | mon | 137 | 1 | 999 | 0 | nonexistent | 1.1 | 93.994 | -36.4 | 4.857 | 5191.0 | no | young | 
50 | blue-collar | married | basic.9y | no | yes | yes | telephone | may | mon | 353 | 1 | 999 | 0 | nonexistent | 1.1 | 93.994 | -36.4 | 4.857 | 5191.0 | no | s

## 4. Summary

In [16]:
print('=' * 60)
print('DATAOPS VALIDATION SUMMARY')
print('=' * 60)
print(f'  Database:  {catalog_db}')
print(f'  Table:     {table_name}')
print(f'  Rows:      {row_count}')
print(f'  Columns:   {len(headers)}')
print(f'  Status:    PASSED')
print('=' * 60)

DATAOPS VALIDATION SUMMARY
  Database:  stack_bank_mktg_dev
  Table:     campaign_results
  Rows:      40846
  Columns:   23
  Status:    PASSED
